# Cassava Leaf Disease — EfficientNet-B0 Inference

**Before running**
1. Add competition data: `cassava-leaf-disease-classification`
2. Upload local `efficientnet_b0.pt` as a Dataset and Add it here
3. Settings: GPU on, **Internet Off**
4. Save Version → Save & Run All → Submit

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from torch import nn
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import efficientnet_b0

IMG_SIZE = 320
BATCH = 32
NUM_CLASSES = 5
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

COMP = Path("/kaggle/input/cassava-leaf-disease-classification")
TEST_DIR = COMP / "test_images"
SAMPLE = COMP / "sample_submission.csv"

# Find the uploaded weight file under /kaggle/input
WEIGHTS = next(Path("/kaggle/input").rglob("efficientnet_b0.pt"))
print("device", DEVICE)
print("weights", WEIGHTS)
print("test_dir", TEST_DIR, "exists", TEST_DIR.exists())

In [ ]:
class LeafDataset(Dataset):
    def __init__(self, image_ids, image_dir, transform):
        self.image_ids = list(image_ids)
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.image_ids)

    def __getitem__(self, index):
        image_id = self.image_ids[index]
        image = Image.open(self.image_dir / image_id).convert("RGB")
        return self.transform(image), image_id


transform = transforms.Compose(
    [
        transforms.Resize(IMG_SIZE + 32),
        transforms.CenterCrop(IMG_SIZE),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ]
)


def build_model():
    # Internet is off: do not download ImageNet weights.
    model = efficientnet_b0(weights=None)
    model.classifier[1] = nn.Linear(model.classifier[1].in_features, NUM_CLASSES)
    state = torch.load(WEIGHTS, map_location=DEVICE)
    model.load_state_dict(state)
    return model.to(DEVICE).eval()


model = build_model()
print("loaded")

In [ ]:
test_ids = sorted(p.name for p in TEST_DIR.glob("*.jpg"))
if not test_ids:
    test_ids = pd.read_csv(SAMPLE)["image_id"].tolist()

loader = DataLoader(
    LeafDataset(test_ids, TEST_DIR, transform),
    batch_size=BATCH,
    shuffle=False,
    num_workers=2,
    pin_memory=True,
)

ids, preds = [], []
with torch.no_grad():
    for images, image_ids in loader:
        logits = model(images.to(DEVICE, non_blocking=True))
        preds.append(logits.argmax(1).cpu().numpy())
        ids.extend(list(image_ids))

submission = pd.DataFrame({"image_id": ids, "label": np.concatenate(preds).astype(int)})
submission.to_csv("submission.csv", index=False)
print(submission.head())
print("rows", len(submission))
print(submission["label"].value_counts().sort_index())